In [1]:
import datetime
import pandas as pd
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from os import listdir
from scipy.interpolate import RegularGridInterpolator
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
## 从NC文件时间转PD时间
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))

## 获取文件夹里所有NC文件
def getfiles(filed):
    files=listdir(filed)
    files.sort()
    files=files[0:]
    files_n=[filed+'/'+i for i in files]
    return files_n

def wr(txt):
    w=open('日志.log','a')
    w.write(f"{txt}")
    w.close()


In [3]:
das1=nc.Dataset('/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/1993_150E_180.nc')
das2=nc.Dataset('/lustre/home/yuhanxue/data/BRAN/ERA5_4Q/1993_0_70W.nc')
lon1=np.array(das1['longitude'])
lon2=np.array(das2['longitude'])
lat_e=np.array(das1['latitude'])[::-1]
lon_e=np.concatenate([lon1[:-1],lon2+360])


In [4]:
print(lat_e)

[20.   20.25 20.5  20.75 21.   21.25 21.5  21.75 22.   22.25 22.5  22.75
 23.   23.25 23.5  23.75 24.   24.25 24.5  24.75 25.   25.25 25.5  25.75
 26.   26.25 26.5  26.75 27.   27.25 27.5  27.75 28.   28.25 28.5  28.75
 29.   29.25 29.5  29.75 30.   30.25 30.5  30.75 31.   31.25 31.5  31.75
 32.   32.25 32.5  32.75 33.   33.25 33.5  33.75 34.   34.25 34.5  34.75
 35.   35.25 35.5  35.75 36.   36.25 36.5  36.75 37.   37.25 37.5  37.75
 38.   38.25 38.5  38.75 39.   39.25 39.5  39.75 40.   40.25 40.5  40.75
 41.   41.25 41.5  41.75 42.   42.25 42.5  42.75 43.   43.25 43.5  43.75
 44.   44.25 44.5  44.75 45.   45.25 45.5  45.75 46.   46.25 46.5  46.75
 47.   47.25 47.5  47.75 48.   48.25 48.5  48.75 49.   49.25 49.5  49.75
 50.   50.25 50.5  50.75 51.   51.25 51.5  51.75 52.   52.25 52.5  52.75
 53.   53.25 53.5  53.75 54.   54.25 54.5  54.75 55.   55.25 55.5  55.75
 56.   56.25 56.5  56.75 57.   57.25 57.5  57.75 58.   58.25 58.5  58.75
 59.   59.25 59.5  59.75 60.  ]


In [5]:
lon_e=np.arange(np.min(lon_e),np.max(lon_e)+1,2)
lat_e=np.arange(np.min(lat_e), np.max(lat_e)+1, 2)

In [6]:
s=np.load('/lustre/home/yuhanxue/data/BRAN/BRAN_lon_lat_level.npz')
lon_B=s['lon']
lat_B=s['lat']
level_B=s['level']

In [7]:
file_add='/lustre/home/yuhanxue/data/BRAN/temp'
file_list=getfiles(file_add)


lon_4=lon_e
lat_4=lat_e
lon_12=lon_B
lat_12=lat_B
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #print(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ThreadPoolExecutor(max_workers=4)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans
def allinone(dat):
    pool = ProcessPoolExecutor(max_workers=36)
    ans=np.array(list(pool.map(list_map,dat)))
    del pool
    return ans

def getneed(year,mon):
    global file_list,file_add
    words=file_add.split('/')
    y=str(year)
    m=int(mon)
    if  f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_2.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_3.nc'
    #print(f'**** {file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy')
def checknpy(arr):
    if arr.shape[1]==33:
        return allinone(arr)
    elif arr.shape[1]==36:
        z=np.zeros(shape=arr.shape)
        z[:,0:11,:,:]=arr[:,0:11,:,:]
        z[:,11:22,:,:]=arr[:,12:23,:,:]
        z[:,22:33,:,:]=arr[:,24:35,:,:]
        return allinone(z[:,:33,:,:])
def readnpy(fi,spe=False):
    if fi.split('.')[-1]=='npy':
        if spe:
            return checknpy(np.load(fi)[:,:-1,:,:])
        else:
            return checknpy(np.load(fi))
    elif fi.split('.')[-1]=='nc':
        das=nc.Dataset(fi)
        return np.array(das[list(das.variables.keys())[0]])
def getnpy(fil):
    try:
        if type(fil)==str:
            return readnpy(fil)
        else:
            a=readnpy(fil[0])
            b=readnpy(fil[1])
            c=readnpy(fil[2])
            if np.sum(a[0]-b[0])==0:
                print(fil)
            return checknpy(np.concatenate([a,b,c],axis=1))
    except:
        print(fil)
        raise ValueError
# for year in range(1993,2023):
#     for i in range(12):
#         print(f'{year}_{i+1}')
#         print(getnpy(getneed(year,i+1)).shape)
# year=1993

Bran_temp=np.concatenate([
    np.concatenate([
        getnpy(getneed(year,i+1)) for i in range(12)
        ],axis=0) for year in range(1993,2023)
    ],axis=0)
np.save('bran_temp.npy',Bran_temp)

lat_4 range:20.00~60.00 | freq:2.0
lon_4 range:150.00~250.00 | freq:2.0
lat_12 range:19.95~64.95 | freq:0.09999847412109375
lon_12 range:149.95~249.95 | freq:0.100006103515625


In [ ]:
np.save('bran_temp.npy',Bran_temp)

In [ ]:
# DEBUG
Bran_temp[Bran_temp<=-1000]=np.nan
test=np.nanmean(np.nanmean(np.nanmean(Bran_temp,axis=-1),axis=-1),axis=-1)
ind=test>=20
times=pd.date_range('1993-01-01','2022-12-31',freq='1D')
times_i=times[ind]
print(np.array(times_i))
plt.plot(test)

In [ ]:
ind=test>=30
times_i=times[ind]
print(np.array(times_i))
plt.plot(test)

In [ ]:
test2=Bran_temp[ind,3,:,:][0,:,:]
print(np.nanmean(Bran_temp[ind,:,:,:][0,:,:,:]))
print(test2)
#test[test<=-1000]=np.nan
plt.contourf(test2)
plt.colorbar()

In [ ]:
file_add='/lustre/home/yuhanxue/data/BRAN/temp'
file_list=getfiles(file_add)


lon_4=lon_e
lat_4=lat_e
lon_12=lon_B
lat_12=lat_B
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #print(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ProcessPoolExecutor(max_workers=3)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans
def allinone(dat):
    pool = ProcessPoolExecutor(max_workers=4)
    ans=np.array(list(pool.map(list_map,dat)))
    del pool
    return ans

def getneed(year,mon):
    global file_list,file_add
    words=file_add.split('/')
    y=str(year)
    m=int(mon)
    if  f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_2.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_3.nc'
    #print(f'**** {file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy')
def checknpy(arr):
    if arr.shape[1]==33:
        return allinone(arr)
    elif arr.shape[1]==36:
        z=np.zeros(shape=arr.shape)
        z[:,0:11,:,:]=arr[:,0:11,:,:]
        z[:,11:22,:,:]=arr[:,12:23,:,:]
        z[:,22:33,:,:]=arr[:,24:35,:,:]
        return allinone(z[:,:33,:,:])
def readnpy(fi,spe=False):
    if fi.split('.')[-1]=='npy':
        if spe:
            return checknpy(np.load(fi)[:,:-1,:,:])
        else:
            return checknpy(np.load(fi))
    elif fi.split('.')[-1]=='nc':
        das=nc.Dataset(fi)
        return np.array(das[list(das.variables.keys())[0]])
def getnpy(fil):
    if type(fil)==str:
        return readnpy(fil)
    else:
        a=readnpy(fil[0])
        b=readnpy(fil[1])
        c=readnpy(fil[2])
        if np.sum(a[0]-b[0])==0:
            print(fil)
        return checknpy(np.concatenate([a,b,c],axis=1))
# for i in range(12):
#     print(getneed(1996,i+1))
# year=1993

Bran_temp=np.concatenate([
    np.concatenate([
        getnpy(getneed(year,i+1)) for i in range(12)
        ],axis=0) for year in range(1993,2023)
    ],axis=0)
file_add='/lustre/home/yuhanxue/data/BRAN/u'
file_list=getfiles(file_add)


lon_4=lon_e
lat_4=lat_e
lon_12=lon_B
lat_12=lat_B
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #print(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ProcessPoolExecutor(max_workers=3)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans
def allinone(dat):
    pool = ProcessPoolExecutor(max_workers=4)
    ans=np.array(list(pool.map(list_map,dat)))
    del pool
    return ans

def getneed(year,mon):
    global file_list,file_add
    words=file_add.split('/')
    y=str(year)
    m=int(mon)
    if  f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_2.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_3.nc'
    #print(f'**** {file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy')
def checknpy(arr):
    if arr.shape[1]==33:
        return allinone(arr)
    elif arr.shape[1]==36:
        z=np.zeros(shape=arr.shape)
        z[:,0:11,:,:]=arr[:,0:11,:,:]
        z[:,11:22,:,:]=arr[:,12:23,:,:]
        z[:,22:33,:,:]=arr[:,24:35,:,:]
        return allinone(z[:,:33,:,:])
def readnpy(fi,spe=False):
    if fi.split('.')[-1]=='npy':
        if spe:
            return checknpy(np.load(fi)[:,:-1,:,:])
        else:
            return checknpy(np.load(fi))
    elif fi.split('.')[-1]=='nc':
        das=nc.Dataset(fi)
        return np.array(das[list(das.variables.keys())[0]])
def getnpy(fil):
    if type(fil)==str:
        return readnpy(fil)
    else:
        a=readnpy(fil[0])
        b=readnpy(fil[1])
        c=readnpy(fil[2])
        if np.sum(a[0]-b[0])==0:
            print(fil)
        return checknpy(np.concatenate([a,b,c],axis=1))
# for i in range(12):
#     print(getneed(1996,i+1))
# year=1993

Bran_u=np.concatenate([
    np.concatenate([
        getnpy(getneed(year,i+1)) for i in range(12)
        ],axis=0) for year in range(1993,2023)
    ],axis=0)

file_add='/lustre/home/yuhanxue/data/BRAN/v'
file_list=getfiles(file_add)


lon_4=lon_e
lat_4=lat_e
lon_12=lon_B
lat_12=lat_B
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #print(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ProcessPoolExecutor(max_workers=3)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans
def allinone(dat):
    pool = ProcessPoolExecutor(max_workers=4)
    ans=np.array(list(pool.map(list_map,dat)))
    del pool
    return ans

def getneed(year,mon):
    global file_list,file_add
    words=file_add.split('/')
    y=str(year)
    m=int(mon)
    if  f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_2.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_3.nc'
    #print(f'**** {file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy')
def checknpy(arr):
    if arr.shape[1]==33:
        return allinone(arr)
    elif arr.shape[1]==36:
        z=np.zeros(shape=arr.shape)
        z[:,0:11,:,:]=arr[:,0:11,:,:]
        z[:,11:22,:,:]=arr[:,12:23,:,:]
        z[:,22:33,:,:]=arr[:,24:35,:,:]
        return allinone(z[:,:33,:,:])
def readnpy(fi,spe=False):
    if fi.split('.')[-1]=='npy':
        if spe:
            return checknpy(np.load(fi)[:,:-1,:,:])
        else:
            return checknpy(np.load(fi))
    elif fi.split('.')[-1]=='nc':
        das=nc.Dataset(fi)
        return np.array(das[list(das.variables.keys())[0]])
def getnpy(fil):
    if type(fil)==str:
        return readnpy(fil)
    else:
        a=readnpy(fil[0])
        b=readnpy(fil[1])
        c=readnpy(fil[2])
        if np.sum(a[0]-b[0])==0:
            print(fil)
        return checknpy(np.concatenate([a,b,c],axis=1))
# for i in range(12):
#     print(getneed(1996,i+1))
# year=1993

Bran_v=np.concatenate([
    np.concatenate([
        getnpy(getneed(year,i+1)) for i in range(12)
        ],axis=0) for year in range(1993,2023)
    ],axis=0)

In [ ]:
np.save('bran_temp.npy',Bran_temp)

In [8]:
file_add='/lustre/home/yuhanxue/data/BRAN/u'
file_list=getfiles(file_add)
lon_4=lon_e
lat_4=lat_e
lon_12=lon_B
lat_12=lat_B
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #print(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ThreadPoolExecutor(max_workers=3)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans
def allinone(dat):
    pool = ProcessPoolExecutor(max_workers=8)
    ans=np.array(list(pool.map(list_map,dat)))
    del pool
    return ans

def getneed(year,mon):
    global file_list,file_add
    words=file_add.split('/')
    y=str(year)
    m=int(mon)
    if  f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_2.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_3.nc'
    #print(f'**** {file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy')
def checknpy(arr):
    if arr.shape[1]==33:
        return allinone(arr)
    elif arr.shape[1]==36:
        z=np.zeros(shape=arr.shape)
        z[:,0:11,:,:]=arr[:,0:11,:,:]
        z[:,11:22,:,:]=arr[:,12:23,:,:]
        z[:,22:33,:,:]=arr[:,24:35,:,:]
        return allinone(z[:,:33,:,:])
def readnpy(fi,spe=False):
    if fi.split('.')[-1]=='npy':
        if spe:
            return checknpy(np.load(fi)[:,:-1,:,:])
        else:
            return checknpy(np.load(fi))
    elif fi.split('.')[-1]=='nc':
        das=nc.Dataset(fi)
        return np.array(das[list(das.variables.keys())[0]])
def getnpy(fil):
    if type(fil)==str:
        return readnpy(fil)
    else:
        a=readnpy(fil[0])
        b=readnpy(fil[1])
        c=readnpy(fil[2])
        if np.sum(a[0]-b[0])==0:
            print(fil)
        return checknpy(np.concatenate([a,b,c],axis=1))
# for i in range(12):
#     print(getneed(1996,i+1))
# year=1993

# Bran_u=np.concatenate([
#     np.concatenate([
#         getnpy(getneed(year,i+1)) for i in range(12)
#         ],axis=0) for year in range(1993,2023)
#     ],axis=0)
# print('u')

def getyear(year):
    return np.concatenate([getnpy(getneed(year,i+1)) for i in range(12)],axis=0)


pool = ProcessPoolExecutor(max_workers=5)
Bran_u=np.concatenate(list(pool.map(getyear,range(1993,2023))),axis=0)
del pool
# Bran_u=np.concatenate(list(map,range(1993,2023)),axis=0)
np.save('bran_u.npy',Bran_u)

lat_4 range:20.00~60.00 | freq:2.0
lon_4 range:150.00~250.00 | freq:2.0
lat_12 range:19.95~64.95 | freq:0.09999847412109375
lon_12 range:149.95~249.95 | freq:0.100006103515625


In [ ]:
np.save('bran_u.npy',Bran_u)

In [9]:
file_add='/lustre/home/yuhanxue/data/BRAN/v'
file_list=getfiles(file_add)
lon_4=lon_e
lat_4=lat_e
lon_12=lon_B
lat_12=lat_B
Lon_4,Lat_4=np.meshgrid(lon_4,lat_4)
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{abs(lat_12[1]-lat_12[0])}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')
def grid(dat):
    global lon_12,lat_12,Lon_4,Lat_4
    #print(dat.shape)
    #f=RegularGridInterpolator((lat_12,lon_12),godas_mlds[1,:,:],bounds_error=False,method='qubic',fill_value=np.nan)
    return RegularGridInterpolator((lat_12,lon_12),dat,bounds_error=False,fill_value=np.nan)((Lat_4,Lon_4))
def list_map(dat):
    pool = ThreadPoolExecutor(max_workers=3)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans
def allinone(dat):
    pool = ProcessPoolExecutor(max_workers=8)
    ans=np.array(list(pool.map(list_map,dat)))
    del pool
    return ans

def getneed(year,mon):
    global file_list,file_add
    words=file_add.split('/')
    y=str(year)
    m=int(mon)
    if  f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}.nc'
    elif f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc' in file_list:
        return f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_1.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_2.nc',f'{file_add}/ocean_{words[-1]}_{y}_{m:02d}_3.nc'
    #print(f'**** {file_add}/ocean_{words[-1]}_{y}_{m:02d}.npy')
def checknpy(arr):
    if arr.shape[1]==33:
        return allinone(arr)
    elif arr.shape[1]==36:
        z=np.zeros(shape=arr.shape)
        z[:,0:11,:,:]=arr[:,0:11,:,:]
        z[:,11:22,:,:]=arr[:,12:23,:,:]
        z[:,22:33,:,:]=arr[:,24:35,:,:]
        return allinone(z[:,:33,:,:])
def readnpy(fi,spe=False):
    if fi.split('.')[-1]=='npy':
        if spe:
            return checknpy(np.load(fi)[:,:-1,:,:])
        else:
            return checknpy(np.load(fi)[:,:33,:,:])
    elif fi.split('.')[-1]=='nc':
        das=nc.Dataset(fi)
        return np.array(das[list(das.variables.keys())[0]])
def getnpy(fil):
    if type(fil)==str:
        return readnpy(fil)
    else:
        a=readnpy(fil[0])
        b=readnpy(fil[1])
        c=readnpy(fil[2])
        if np.sum(a[0]-b[0])==0:
            print(fil)
        return checknpy(np.concatenate([a,b,c],axis=1))
# for i in range(12):
#     print(getneed(1996,i+1))
# year=1993

# Bran_u=np.concatenate([
#     np.concatenate([
#         getnpy(getneed(year,i+1)) for i in range(12)
#         ],axis=0) for year in range(1993,2023)
#     ],axis=0)
# print('u')

def getyear(year):
    return np.concatenate([getnpy(getneed(year,i+1)) for i in range(12)],axis=0)


pool = ProcessPoolExecutor(max_workers=5)
Bran_v=np.concatenate(list(pool.map(getyear,range(1993,2023))),axis=0)
del pool
# Bran_u=np.concatenate(list(map,range(1993,2023)),axis=0)
# np.save('bran_u.npy',Bran_u)

# getnpy(getneed(1995,11)).shape

lat_4 range:20.00~60.00 | freq:2.0
lon_4 range:150.00~250.00 | freq:2.0
lat_12 range:19.95~64.95 | freq:0.09999847412109375
lon_12 range:149.95~249.95 | freq:0.100006103515625


In [ ]:
np.save('bran_v.npy',Bran_v)

In [ ]:
Bran_temp=np.load('/lustre/home/yuhanxue/bran_temp.npy')
Bran_u=np.load('/lustre/home/yuhanxue/bran_u.npy')
Bran_v=np.load('/lustre/home/yuhanxue/bran_v.npy')

In [ ]:
for i in range(12):
    print(i,getnpy(getneed(1995,i+1)).shape)

In [ ]:
test=np.load(getneed(1995,11))

In [ ]:
np.unique(test[0,:,0,0])

In [ ]:
test[0,:,0,0]

In [ ]:
plt.plot(test[0,:,0,0])

getyear(2003).shape

In [ ]:
pool = ProcessPoolExecutor(max_workers=8)
demo=np.array(list(pool.map(list_map,getnpy(getneed(1993,1)))))
demo=np.array(list(pool.map(list_map,getnpy(getneed(1993,1)))))
del pool

In [ ]:
plt.contourf(Lon_4,Lat_4,demo[0,0,:,:])

In [ ]:
getnpy(getneed(1993,1)).shape

In [ ]:

#test=getnpy(getneed(1996,1))
#[getneed(year,i+1) for i in range(12)]
for year in range(1993,2023):
    for i in range(12):
        test=getnpy(getneed(year,i+1))

In [ ]:
li=getneed(1996,1)
getnpy(li[0])

In [ ]:
year=2016
#test=getnpy(getneed(year,5))
#[getneed(year,i+1) for i in range(12)]
test=getnpy([getneed(year,i+1) for i in range(12)][7])
[getneed(year,i+1) for i in range(12)][7]

In [ ]:
te

In [ ]:
checknpy(test).shape

In [ ]:
np.delete(test,).shape

In [ ]:
getneed(year,1)

In [ ]:
plt.plot(checknpy(test)[0,:,60,0])

In [ ]:
year=1996
[(getneed(year,i+1)) for i in range(12)]

In [ ]:
test=getnpy([getneed(year,i+1) for i in range(12)][1])
#test2=getnpy([getneed(year,i+1) for i in range(12)][0])

In [ ]:
plt.plot(test[0,:,0,0])

In [ ]:
plt.plot(getnpy([getneed(year,i+1) for i in range(12)][0][1])[0,:,0,0])

In [ ]:
import sys

if __name__ == "__main__":
    # 接收命令行参数
    myarg1 = sys.argv[1]
    myarg2 = sys.argv[2]
    myarg3 = sys.argv[3]

    # 打印参数
    print("Argument 1:", myarg1)
    print("Argument 2:", myarg2)
    print("Argument 3:", myarg3)

In [ ]:
data = loadmat('filename.mat')